In [1]:
from PIL import Image #open imgs for v1, v2
import matplotlib.pyplot as plt #plot imgs
import cv2 #for v3
import os #file and image management
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import re #regular expressions
import torch
import numpy as np
import sys

In [2]:
import transformers

In [2]:
from transformers import pipeline #easy pipeline for DA models



In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu" #set device as GPU if available

models = {
    "V1": "LiheYoung/depth-anything-base-hf",
    "V2": "depth-anything/Depth-Anything-V2-base-hf",
    # "V3": "qualcomm/Depth-Anything-V3"
}


    
pipelines = {}
print("\nLoading models")
for version, repo in models.items():
    try:
        pipelines[version] = pipeline(task="depth-estimation", model=repo, device=device, torch_dtype=torch.float32)
        print(f" {version} loaded successfully.")
    except Exception as e:
        print(f" Couldn't Load {version}.")


# v3_model_path = os.path.join(v3_root, "checkpoints", "DA3NESTED-BASE") 




Loading models


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


 V1 loaded successfully.


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

 V2 loaded successfully.


In [4]:

current_dir = os.getcwd()
v3_folder = os.path.join(current_dir, "depth-anything-3")
v3_src = os.path.join(v3_folder, "src")

if v3_src not in sys.path:
    sys.path.insert(0, v3_src)

# set device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- loading depth anything v3 local (base) ---")
try:
    # now that the folder is next to the notebook, this should work
    from depth_anything_3.api import DepthAnything3
    
    # attempt to load the base model
    # note: if you get a 401 error, we will need to point to a local .pth file
    model = DepthAnything3.from_pretrained("depth-anything/DA3-BASE")
    model = model.to(device=torch.device(device))
    
    pipelines["V3"] = model
    print(" v3 base loaded successfully from local folder.")
except ModuleNotFoundError as e:
    print(f" import error: {e}. check if 'depth-anything-3/src/depth_anything_3' exists.")
except Exception as e:
    print(f" load failed: {e}")

--- loading depth anything v3 local (base) ---
[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.7.1+cu126 with CUDA 1208 (you have 2.7.1+cu118)
    Python  3.9.13 (you have 3.10.10)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


[INFO ] using MLP layer as FFN
 v3 base loaded successfully from local folder.


In [6]:
def compare_single_image(image_path, save_path=None):
    """
    runs all available models (v1, v2, v3) and plots them with unified color logic
    """
    try:
        available_models = sorted(pipelines.keys())
        if not available_models: return

        # setup naming
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        clean_name = re.sub(r'\d+', '', base_name).replace('_', ' ').strip().title()

        img_pil = Image.open(image_path).convert("RGB")
        img_np = np.array(img_pil)
        
        results = {}
        for version in available_models:
            model = pipelines[version]
            if version == "V3":
                prediction = model.inference([img_np])
                # v3 output is often disparity (higher = closer)
                # we keep it raw here and handle logic in the plotting loop
                results[version] = prediction.depth[0]
            else:
                res = model(img_pil, timeout=None)
                results[version] = res["predicted_depth"].squeeze().cpu().numpy()

        num_plots = len(results) + 1
        fig, axes = plt.subplots(1, num_plots, figsize=(5 * num_plots, 5))
        if num_plots == 1: axes = [axes]
        fig.suptitle(clean_name, fontsize=14, fontweight='bold', y=0.98)

        # 0. original image
        axes[0].imshow(img_pil)
        axes[0].set_title("Original Image", fontsize=10, fontweight='bold')
        axes[0].axis('off')

        # 1-N. depth maps
        for i, version in enumerate(available_models):
            depth_map_ptr = np.array(results[version])
            
            # basic normalization: 0 to 1
            d_min, d_max = depth_map_ptr.min(), depth_map_ptr.max()
            depth_norm = (depth_map_ptr - d_min) / (d_max - d_min + 1e-8)

            # logical check for v3
            # if the floor/bottom of the ames room is darker than the ceiling, it's inverted
            # standard: closer objects (bottom of image usually) should be lighter (yellow/white)
            if version == "V3":
                # we force v3 to match v1/v2 orientation
                # usually v1/v2: close = high value (light)
                # if your v3 is showing close = dark, we flip it
                # current evidence shows v3 needs flipping to match v1/v2 behavior
                depth_norm = 1.0 - depth_norm

            axes[i+1].imshow(depth_norm, cmap='inferno', interpolation="bilinear")
            axes[i+1].set_title(f"Depth Anything {version}", fontsize=10, fontweight='bold')
            axes[i+1].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.92])

        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()

    except Exception as e:
        print(f" error in compare_single_image: {e}")

def process_dataset(input_folder="3D_illusions", output_folder="results"):
    """
    automates processing for the entire directory structure
    """
    # final check before starting
    print(f"models currently loaded: {list(pipelines.keys())}")
    # print(f"starting batch processing...")
    
    total = 0
    if not os.path.exists(input_folder):
        print(f" error: input folder '{input_folder}' not found.")
        return

    for root, dirs, files in os.walk(input_folder):
        for filename in files:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, filename)
                
                # maintain subdirectory structure
                rel_path = os.path.relpath(root, input_folder)
                target_dir = os.path.join(output_folder, rel_path)
                os.makedirs(target_dir, exist_ok=True)
                
                save_path = os.path.join(target_dir, f"cmp_{filename}")
                
                print(f" processing -> {filename}")
                compare_single_image(img_path, save_path=save_path)
                total += 1

    print(f"dataset processing complete. total images: {total}")

# execute
# process_dataset()

In [ ]:
process_dataset(input_folder="synthetic data/corridor_illusion", output_folder="synthetic_results") #for the corridor illusion

models currently loaded: ['V1', 'V2', 'V3']
 processing -> base_image.png
[INFO ] Processed Images Done taking 0.028921842575073242 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.06757259368896484 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997781753540039 seconds
 processing -> cube_walls.jpg
[INFO ] Processed Images Done taking 0.013963460922241211 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.08509469032287598 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997304916381836 seconds
 processing -> cylinder_pattern.png
[INFO ] Processed Images Done taking 0.026926517486572266 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.07308721542358398 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997781753540039 seconds
 processing -> cylinder_pattern_bordered.png
[INFO ] Processed Images Done taking 0.012859106063842773 seconds. Shape:  torch

In [20]:
process_dataset(input_folder="Minecraft Experiments", output_folder="Minecraft_Results") #Minecraft Experiments

models currently loaded: ['V1', 'V2', 'V3']
 processing -> flat_grid.png
[INFO ] Processed Images Done taking 0.019443511962890625 seconds. Shape:  torch.Size([1, 3, 266, 504])
[INFO ] Model Forward Pass Done. Time: 0.05038952827453613 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0 seconds
 processing -> hallway.jpg
[INFO ] Processed Images Done taking 0.013962984085083008 seconds. Shape:  torch.Size([1, 3, 280, 504])
[INFO ] Model Forward Pass Done. Time: 0.08441877365112305 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0 seconds
 processing -> TOP_BOTTOM_bias.png
[INFO ] Processed Images Done taking 0.017952680587768555 seconds. Shape:  torch.Size([1, 3, 266, 504])
[INFO ] Model Forward Pass Done. Time: 0.07583308219909668 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0009975433349609375 seconds
 processing -> grid_bias.png
[INFO ] Processed Images Done taking 0.02144026756286621 seconds. Shape:  torch.Size([1, 3, 266, 504])
[INFO ] Model Forward Pass Done

In [14]:
import cv2
import numpy as np
import os
import csv

rectangles = []
drawing = False
ix, iy = -1, -1
img_with_commands = None
raw_img = None
scale_ratio = 1.0
display_offset = 0
orig_offset = 0
banner_h = 40
num_panels = 4
last_category = ""

def draw_rect(event, x, y, flags, param):
    """handles click-and-drag to draw bounding boxes."""
    global ix, iy, drawing, img_with_commands, rectangles, display_offset, banner_h, num_panels
    
    if event == cv2.EVENT_LBUTTONDOWN:
        if y < banner_h: return # ignore clicks on banner
        if x > display_offset: return # force drawing on panel 1 only
        drawing = True
        ix, iy = x, y
        
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            temp_img = img_with_commands.copy()
            for i in range(num_panels):
                nx1, nx2 = ix + i * display_offset, x + i * display_offset
                cv2.rectangle(temp_img, (nx1, iy), (nx2, y), (0, 255, 255), 1)
            cv2.imshow("tfg_lab", temp_img)
            
    elif event == cv2.EVENT_LBUTTONUP:
        if drawing:
            drawing = False
            x1, x2 = min(ix, x), max(ix, x)
            y1, y2 = min(iy, y), max(iy, y)
            
            # prevent accidental tiny clicks
            if x2 - x1 > 2 and y2 - y1 > 2:
                rectangles.append((x1, y1, x2, y2))
                for i in range(num_panels):
                    nx1, nx2 = x1 + i * display_offset, x2 + i * display_offset
                    cv2.rectangle(img_with_commands, (nx1, y1), (nx2, y2), (0, 255, 0), 2)
                    cv2.putText(img_with_commands, f"roi{len(rectangles)}", (nx1, y1-5), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                cv2.imshow("tfg_lab", img_with_commands)

def run_tfg_experiment(results_path="results", output_path="ROI_results"):
    global rectangles, img_with_commands, raw_img, scale_ratio, display_offset, orig_offset, banner_h, num_panels, last_category
    
    os.makedirs(output_path, exist_ok=True)
    csv_file = os.path.join(output_path, "tfg_roi_metrics.csv")
    
    if not os.path.exists(csv_file):
        with open(csv_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["category", "image", "model", "roi_id", "area_px", "mean_intensity", "gap_with_prev_roi"])

    for category in sorted(os.listdir(results_path)):
        cat_path = os.path.join(results_path, category)
        if not os.path.isdir(cat_path): continue

        if category != last_category:
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["---", f"CATEGORY: {category.upper()}", "---", "---", "---", "---", "---"])
            last_category = category

        valid_ext = ('.png', '.jpg', '.jpeg', '.bmp')
        image_files = [f for f in os.listdir(cat_path) if f.lower().endswith(valid_ext)]

        for filename in image_files:
            print(f"\n--- analyzing: {filename} ---")
            
            target_path = os.path.join(cat_path, filename)
            raw_img = cv2.imread(target_path, cv2.IMREAD_GRAYSCALE)
            if raw_img is None: continue
            
            h, w_total = raw_img.shape
            
            # auto-detect offset logic
            analysis_region = raw_img[int(h*0.15):int(h*0.95), :]
            _, thresh = cv2.threshold(analysis_region, 240, 255, cv2.THRESH_BINARY_INV)
            col_sums = np.sum(thresh, axis=0)
            content_cols = np.where(col_sums > 20)[0]
            
            if len(content_cols) > 0:
                jumps = np.where(np.diff(content_cols) > 15)[0]
                clusters = []
                start_idx = 0
                for j in jumps:
                    clusters.append((content_cols[start_idx], content_cols[j]))
                    start_idx = j + 1
                clusters.append((content_cols[start_idx], content_cols[-1]))
                
                if len(clusters) >= 4:
                    centers = [(start + end) / 2.0 for start, end in clusters]
                    orig_offset = int(np.mean(np.diff(centers[:4])))
                else:
                    orig_offset = w_total // num_panels
            else:
                orig_offset = w_total // num_panels
            
            max_w, max_h = 1800, 800
            scale_ratio = min(max_w/w_total, max_h/h, 1.0)
            new_w, new_h = int(w_total * scale_ratio), int(h * scale_ratio)
            display_offset = int(orig_offset * scale_ratio)
            
            base_display = cv2.cvtColor(cv2.resize(raw_img, (new_w, new_h)), cv2.COLOR_GRAY2BGR)
            
            img_with_commands = np.zeros((new_h + banner_h, new_w, 3), dtype=np.uint8)
            img_with_commands[banner_h:, :] = base_display
            
            cv2.putText(img_with_commands, "keys: [s] save | [r] reset | [n] next  --  DRAG to draw rectangles on panel 1", 
                        (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            rectangles = []
            cv2.namedWindow("tfg_lab", cv2.WINDOW_AUTOSIZE)
            cv2.setMouseCallback("tfg_lab", draw_rect)
            
            break_all = False
            while True:
                cv2.imshow("tfg_lab", img_with_commands)
                key = cv2.waitKey(1) & 0xFF
                
                if key == ord('s'):
                    out_img = cv2.cvtColor(raw_img, cv2.COLOR_GRAY2BGR)
                    
                    with open(csv_file, mode='a', newline='') as f:
                        writer = csv.writer(f)
                        means = {"orig": [], "v1": [], "v2": [], "v3": []}
                        
                        for i, (px1, py1, px2, py2) in enumerate(rectangles):
                            # map back to original resolution coordinates
                            true_y1, true_y2 = py1 - banner_h, py2 - banner_h
                            ox1, oy1 = int(px1 / scale_ratio), int(true_y1 / scale_ratio)
                            ox2, oy2 = int(px2 / scale_ratio), int(true_y2 / scale_ratio)
                            
                            area = (ox2 - ox1) * (oy2 - oy1)
                            
                            # extract roi blocks
                            roi_orig = raw_img[oy1:oy2, ox1:ox2]
                            roi_v1 = raw_img[oy1:oy2, ox1 + orig_offset : ox2 + orig_offset]
                            roi_v2 = raw_img[oy1:oy2, ox1 + 2 * orig_offset : ox2 + 2 * orig_offset]
                            roi_v3 = raw_img[oy1:oy2, ox1 + 3 * orig_offset : ox2 + 3 * orig_offset]
                            
                            # calculate means
                            mean_orig = round(np.mean(roi_orig), 2)
                            mean_v1 = round(np.mean(roi_v1), 2)
                            mean_v2 = round(np.mean(roi_v2), 2)
                            mean_v3 = round(np.mean(roi_v3), 2)
                            
                            means["orig"].append(mean_orig)
                            means["v1"].append(mean_v1)
                            means["v2"].append(mean_v2)
                            means["v3"].append(mean_v3)
                            
                            # gap against previous ROI (pairwise logic)
                            gap_orig = round(abs(mean_orig - means["orig"][i-1]), 2) if i % 2 == 1 else 0.0
                            gap_v1 = round(abs(mean_v1 - means["v1"][i-1]), 2) if i % 2 == 1 else 0.0
                            gap_v2 = round(abs(mean_v2 - means["v2"][i-1]), 2) if i % 2 == 1 else 0.0
                            gap_v3 = round(abs(mean_v3 - means["v3"][i-1]), 2) if i % 2 == 1 else 0.0
                            
                            writer.writerow([category, filename, "orig", i+1, area, mean_orig, gap_orig])
                            writer.writerow([category, filename, "v1", i+1, area, mean_v1, gap_v1])
                            writer.writerow([category, filename, "v2", i+1, area, mean_v2, gap_v2])
                            writer.writerow([category, filename, "v3", i+1, area, mean_v3, gap_v3])
                            
                            # draw rectangles and mean values on final image
                            for j, m_val in enumerate([mean_orig, mean_v1, mean_v2, mean_v3]):
                                curr_x1 = ox1 + j * orig_offset
                                curr_x2 = ox2 + j * orig_offset
                                cv2.rectangle(out_img, (curr_x1, oy1), (curr_x2, oy2), (0, 255, 0), 2)
                                cv2.putText(out_img, f"r{i+1}: {m_val}", (curr_x1, oy1-10), 
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    
                    out_dir = os.path.join(output_path, category)
                    os.makedirs(out_dir, exist_ok=True)
                    cv2.imwrite(os.path.join(out_dir, f"analisis_{filename}"), out_img)
                    print("  -> saved successfully.")
                    break
                
                elif key == ord('r'):
                    img_with_commands[banner_h:, :] = base_display
                    img_with_commands[:banner_h, :] = 0
                    cv2.putText(img_with_commands, "keys: [s] save | [r] reset | [n] next  --  DRAG to draw rectangles on panel 1", 
                                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                    rectangles = []
                
                elif key == ord('n'):
                    break
                
                elif key == 27:
                    break_all = True
                    break
            
            if break_all: break
        if break_all: break

    cv2.destroyAllWindows()

# run_tfg_experiment()
# run_tfg_experiment(results_path="synthetic_results", output_path="Synthetic_ROI_results")
run_tfg_experiment(results_path="Minecraft_Results", output_path="Minecraft_ROI_results")


--- analyzing: cmp_TOP_BOTTOM_bias.png ---
  -> saved successfully.
